In [ ]:
print("GRACE")

In [ ]:
%pip install openpyxl

In [ ]:
%pip install pandas openpyxl pdfkit xlsxwriter


In [23]:
from PIL import Image, ImageDraw, ImageFont
import platform
import os 

def generate_second_page_with_info(address, assessment_date, next_assessment_date, assessor, responsible_person, form_id):
    # Load the image
    image_path = "second_page_template.png"
    # Define the output path with dynamic form_id
    output_path = f"downloads/second_page/second_page_{form_id}.pdf"

    # Check if the directory exists, and create it if it doesn't
    directory = os.path.dirname(output_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    
    image = Image.open(image_path)

    # Create a drawing object
    draw = ImageDraw.Draw(image)

    # Specify the correct font path for Mac
    font_path_mac = "/Library/Fonts/Arial Unicode.ttf"  # Correct font for Mac

    # Try to detect if the system is Mac or Linux
    system = platform.system()

    if system == 'Darwin':  # For macOS
        font_path = font_path_mac
    elif system == 'Linux':  # For Linux
        font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"  # Example for Linux
    else:
        font_path = "/path/to/your/font.ttf"  # Provide your custom font

    # Load the font with a larger size
    font = ImageFont.truetype(font_path, size=30)  # Adjust font size as needed

    # List of texts and their corresponding positions
    texts = [
        (f"{address}", (200, 550)),   
        (f"{assessment_date}", (200, 670)),
        (f"{next_assessment_date}", (200, 780)),
        (f"{assessor}", (200, 900)),
        (f"{responsible_person}", (200, 1100)),
    ]

    # Set text color to black
    text_color = (0, 0, 0)  # Black color (R, G, B)

    # Add text to the image
    for text, position in texts:
        draw.text(position, text, fill=text_color, font=font)

    # Convert the image to RGB if it's not already in that mode
    image = image.convert("RGB")

    # Save the image as a PDF
    image.save(output_path, "PDF")


    print(f"Image saved at {output_path}")



Image saved at downloads/second_page/second_page_16.pdf


In [1]:
from PIL import Image, ImageDraw, ImageFont
import os
from datetime import datetime
import reportlab
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

from PIL import Image, ImageDraw, ImageFont
import os

import platform

In [4]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas
from PIL import Image
import os

def reference_images_to_pdf(form_id):
    # Path to the folder containing images
    image_folder = f"uploads/{form_id}"
    
    # Output PDF path
    output_pdf_path = f"downloads/reference_pictures_{form_id}.pdf"
    
    # Ensure downloads directory exists
    os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
    
    # Create PDF canvas
    c = canvas.Canvas(output_pdf_path, pagesize=A4)
    width, height = A4
    
    # Calculate image size (1/3rd of A4 width minus padding)
    img_size = (width - 2 * inch - 2 * 0.5 * inch) / 3  # Adjusted for margins and padding
    
    # Margins and padding
    margin_x = inch  # Left and right margins
    margin_y = 0.5 * inch  # Reduced top and bottom margins
    vertical_padding = 0.5 * inch  # Space between rows of images
    horizontal_padding = 0.5 * inch  # Space between columns of images
    
    # White background for the entire page
    c.setFillColorRGB(1, 1, 1)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    
    # Header for Reference Images
    header_text = "Reference Images"
    header_font_size = 20
    header_font = "Helvetica-Bold"
    
    # Calculate header dimensions
    text_width = c.stringWidth(header_text, header_font, header_font_size)
    text_height = header_font_size  # Approximate height of the text
    header_padding = 0.2 * inch  # Padding around the text
    
    # Dark red background for the header
    c.setFillColorRGB(0.5, 0, 0)  # Dark red color
    c.rect(
        (width - text_width) / 2 - header_padding,  # X position (centered)
        height - margin_y - text_height - header_padding,  # Y position
        text_width + 2 * header_padding,  # Width of the background
        text_height + 2 * header_padding,  # Height of the background
        fill=1,
        stroke=0
    )
    
    # White text for the header
    c.setFillColorRGB(1, 1, 1)  # White color
    c.setFont(header_font, header_font_size)
    c.drawCentredString(width / 2, height - margin_y - text_height, header_text)
    
    # Get list of image files
    image_files = [f for f in os.listdir(image_folder) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    
    # Track vertical position and page number
    y_position = height - margin_y - 1.5 * inch - (text_height + 2 * header_padding)  # Adjusted for header
    page_number = 1
    x_positions = [
        margin_x, 
        margin_x + img_size + horizontal_padding, 
        margin_x + 2 * (img_size + horizontal_padding)
    ]
    current_column = 0
    
    # Add images to PDF
    for filename in image_files:
        # Full path to image
        image_path = os.path.join(image_folder, filename)
        
        # Open image to get dimensions
        img = Image.open(image_path)
        img_width, img_height = img.size
        
        # Calculate scaled image size maintaining aspect ratio
        aspect_ratio = img_height / img_width
        scaled_height = img_size * aspect_ratio
        
        # Check if we need a new page
        if y_position - scaled_height < margin_y:
            c.showPage()
            page_number += 1
            y_position = height - margin_y - 1.5 * inch - (text_height + 2 * header_padding)  # Adjusted for header
            current_column = 0
        
        # Draw image
        c.drawImage(image_path, x_positions[current_column], y_position - scaled_height, 
                    width=img_size, height=scaled_height, preserveAspectRatio=True)
        
        # Move to next column/row
        current_column += 1
        if current_column > 2:
            current_column = 0
            y_position -= scaled_height + vertical_padding
    
    # Save PDF
    c.save()

In [5]:
reference_images_to_pdf(22)